# q3: devices
do desktop, mobile, and tablet visitors convert differently, and at which funnel step?
cumulative funnel by device (q3 in sql/05_devices.sql).


In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)


In [2]:
sql = """
select
    device,
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(added_to_cart or began_checkout or purchased) as reached_cart,
    countif(began_checkout or purchased) as reached_checkout,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
group by device
order by sessions desc
"""
devices = client.query(sql).to_dataframe()
devices

,device,sessions,reached_view,reached_cart,reached_checkout,purchases,revenue
0,desktop,208942,44981,11756,6392,2299,178182.0
1,mobile,143185,30628,8146,4474,1663,124160.0
2,tablet,8002,1705,431,240,82,5298.0


In [3]:
devices["view_rate"] = (devices["reached_view"] / devices["sessions"]).round(4)
devices["cart_rate"] = (devices["reached_cart"] / devices["reached_view"]).round(4)
devices["checkout_rate"] = (devices["reached_checkout"] / devices["reached_cart"]).round(4)
devices["purchase_rate"] = (devices["purchases"] / devices["reached_checkout"]).round(4)
devices["conversion_rate"] = (devices["purchases"] / devices["sessions"]).round(4)
devices["revenue_per_session"] = (devices["revenue"] / devices["sessions"]).round(2)
print(devices["sessions"].sum(), devices["purchases"].sum(), devices["revenue"].sum())
devices

360129 4044 307640.0


,device,sessions,reached_view,reached_cart,reached_checkout,purchases,revenue,view_rate,cart_rate,checkout_rate,purchase_rate,conversion_rate,revenue_per_session
0,desktop,208942,44981,11756,6392,2299,178182.0,0.2153,0.2614,0.5437,0.3597,0.011,0.85
1,mobile,143185,30628,8146,4474,1663,124160.0,0.2139,0.266,0.5492,0.3717,0.0116,0.87
2,tablet,8002,1705,431,240,82,5298.0,0.2131,0.2528,0.5568,0.3417,0.0102,0.66


In [4]:
devices.to_csv(output_dir/"q3_devices.csv", index=False)